<a href="https://colab.research.google.com/github/johanhoffman/DD2365_FEniCSx/blob/main/PeriodicBC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# About the code

In [ ]:
# Copyright (C) 2020-2026 Johan Hoffman (jhoffman@kth.se)
#
# This file is part of the course DD2365 Advanced Computation in Fluid Mechanics
# KTH Royal Institute of Technology, Stockholm, Sweden
#
# This is free software: you can redistribute it and/or modify
# it under the terms of the GNU Lesser General Public License as published by
# the Free Software Foundation, either version 3 of the License, or
# (at your option) any later version.
#
# This template is maintained by Johan Hoffman
# Please report problems to jhoffman@kth.se

# Setup environment

In [ ]:
# --- canonical: bootstrap v2 ---
import os as _os
_os.environ.setdefault("OMP_NUM_THREADS", "1")
_os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
_os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
_os.environ.setdefault("MKL_NUM_THREADS", "1")

import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
# --- canonical: periodic_restriction v2 ---
"""
Periodic boundary condition via algebraic restriction matrix (serial only).

periodic_restriction(V, x_target, x_source, tol=1e-12) -> (P, n_red)
  P : PETSc AIJ matrix (n_full x n_red).  Each non-source dof maps to itself
      (identity column), each source dof maps to its matched target dof column.
  n_red : number of columns in P = number of independent dofs after periodicity.

reduce_system(A, b, P) -> (Ar, br)
  Ar = P^T A P  (via MatPtAP),  br = P^T b.

expand_solution(x_r, P) -> x_full
  x_full = P x_r.  Source dofs automatically receive the value of their target.

dof_map(P) -> np.ndarray shape (n_full,)
  full_to_red[i] = column index (reduced dof) for full dof i.
  Call once at setup; use numpy indexing to map Dirichlet full dofs to reduced.

apply_dirichlet_reduced(Ar, br, bc_red_dofs, bc_red_vals)
  Zero rows/cols, unit diagonal, lifted RHS in the reduced system.
  Pass deduplicated reduced-space dofs and their prescribed values.
"""

import numpy as np
from petsc4py import PETSc


def periodic_restriction(V, x_target, x_source, tol=1e-12):
    """
    Build restriction matrix P for x-periodicity; assert serial.

    Source nodes (x ≈ x_source) are identified with target nodes (x ≈ x_target)
    by matching the remaining coordinates within tol, per block component.
    Corner nodes that are both periodic and Dirichlet must carry the same
    prescribed value on both sides (checked in apply_dirichlet_reduced by the
    fact that both map to the same reduced dof).

    Returns (P, n_red).
    """
    assert V.mesh.comm.size == 1, "periodic_restriction: serial only"

    bs     = V.dofmap.bs                         # block size (1 scalar, 2 vector, …)
    coords = V.tabulate_dof_coordinates()        # (n_nodes, 3)  one row per block-node
    n_nodes = coords.shape[0]
    n_full  = n_nodes * bs

    # Identify source nodes (x ≈ x_source) and target nodes (x ≈ x_target)
    src_idx = np.where(np.abs(coords[:, 0] - x_source) < tol)[0]
    tgt_idx = np.where(np.abs(coords[:, 0] - x_target) < tol)[0]
    tgt_y   = coords[tgt_idx, 1]

    # Match each source node to the target node with the same y-coordinate
    node_pair = {}                               # source_node -> target_node
    for s in src_idx:
        diff = np.abs(tgt_y - coords[s, 1])
        j    = int(np.argmin(diff))
        if diff[j] > tol:
            raise RuntimeError(
                f"periodic_restriction: no target match for source node at y={coords[s,1]:.6g}"
            )
        node_pair[int(s)] = int(tgt_idx[j])

    src_set = set(node_pair)

    # Assign reduced node indices: non-source nodes in order, source -> target's index
    node_to_red = {}
    r = 0
    for n in range(n_nodes):
        if n not in src_set:
            node_to_red[n] = r; r += 1
    for s, t in node_pair.items():
        node_to_red[s] = node_to_red[t]          # source shares target's reduced index
    n_red = r * bs

    # Map every full scalar dof to its reduced dof (blocked layout: dof nd*bs+c is node nd, component c)
    full_to_red = np.empty(n_full, dtype=np.int64)
    for nd in range(n_nodes):
        for comp in range(bs):
            full_to_red[nd * bs + comp] = node_to_red[nd] * bs + comp

    # Build P: one nonzero per row
    P = PETSc.Mat().createAIJ([n_full, n_red], nnz=1, comm=PETSc.COMM_SELF)
    P.setUp()
    for row in range(n_full):
        P.setValue(int(row), int(full_to_red[row]), 1.0)
    P.assemblyBegin()
    P.assemblyEnd()
    return P, n_red


def reduce_system(A, b, P):
    """Compute Ar = P^T A P (MatPtAP) and br = P^T b."""
    Ar = A.ptap(P)
    br = P.createVecRight()          # length n_red
    P.multTranspose(b, br)           # br = P^T b
    return Ar, br


def expand_solution(x_r, P):
    """Compute x_full = P x_r."""
    x_full = P.createVecLeft()       # length n_full
    P.mult(x_r, x_full)
    return x_full


def dof_map(P):
    """
    Return full_to_red array: full_to_red[i] = reduced dof index for full dof i.
    O(n_full); call once at setup, not in the time loop.
    """
    n_full, _ = P.getSize()
    f2r = np.empty(n_full, dtype=np.int32)
    for i in range(n_full):
        cols, _ = P.getRow(i)
        f2r[i] = int(cols[0])
    return f2r


def apply_dirichlet_reduced(Ar, br, bc_red_dofs, bc_red_vals):
    """
    Apply Dirichlet BCs in the reduced system.
    Zero rows/cols, unit diagonal, lifted RHS (standard penalty-free approach).

    bc_red_dofs : list/array of reduced-space dof indices (deduplicated).
    bc_red_vals : corresponding prescribed values.
    """
    n_red = Ar.getSize()[0]

    # Dirichlet value vector in reduced space
    u_D = PETSc.Vec().createSeq(n_red, comm=PETSc.COMM_SELF)
    u_D.set(0.0)
    for r, v in zip(bc_red_dofs, bc_red_vals):
        u_D.setValue(int(r), float(v))
    u_D.assemblyBegin(); u_D.assemblyEnd()

    # zeroRowsColumns zeros row r and col r, sets diag=1, and lifts br:
    #   br[j] -= Ar[j,r] * u_D[r]  for j != r,  br[r] = u_D[r] * diag
    Ar.zeroRowsColumns([int(r) for r in bc_red_dofs], diag=1.0, x=u_D, b=br)
# --- end canonical: periodic_restriction ---

In [ ]:
# --- canonical: time_step v1 ---
import numpy as np


def time_step(msh, U_ref, C_CFL=0.5):
    """Return C_CFL * h_min / U_ref (CFL-based time step)."""
    tdim = msh.topology.dim
    n_cells = msh.topology.index_map(tdim).size_local
    h_min = float(msh.h(tdim, np.arange(n_cells, dtype=np.int32)).min())
    return C_CFL * h_min / U_ref
# --- end canonical: time_step ---

In [ ]:
# --- canonical: plot_helpers v3 ---
import numpy as np
import matplotlib.pyplot as plt
import basix.ufl as _bufl
from dolfinx.fem import functionspace, Function


def plot_mesh(msh, title="Mesh"):
    """Plot a 2-D triangular mesh using matplotlib triplot."""
    msh.topology.create_connectivity(msh.topology.dim, 0)
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.triplot(x[:, 0], x[:, 1], gdm, linewidth=0.3, color="k")
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def _p1_scalar_space(msh):
    return functionspace(msh, ("Lagrange", 1))


def _scatter_p1_scalar(f1):
    """Scatter a P1 scalar Function values to geometry nodes.

    Returns (x, geometry_connectivity, node_values).
    """
    V = f1.function_space
    msh = V.mesh
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    ldm = V.dofmap.list
    vals = np.zeros(x.shape[0])
    vals[gdm.ravel()] = f1.x.array.real[ldm.ravel()]
    return x, gdm, vals


def plot_p1(u, title="Solution"):
    """Plot a P1 scalar Function using matplotlib tripcolor (Gouraud shading)."""
    plot_scalar(u, title=title)


def plot_scalar(f, title="Scalar field", ax=None):
    """Plot any scalar Lagrange Function via tripcolor (interpolates to P1 if needed).

    Parameters
    ----------
    f : dolfinx.fem.Function   scalar; any Lagrange degree
    title : str
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    if el.degree == 1 and el.reference_value_shape == ():
        f1 = f
    else:
        f1 = Function(_p1_scalar_space(msh))
        f1.interpolate(f)
    x, gdm, vals = _scatter_p1_scalar(f1)
    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, vals, shading="gouraud")
    plt.colorbar(tc, ax=ax)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()


def plot_vector(u, title="Vector field", quiver=True, quiver_stride=8, ax=None):
    """Plot a 2-D vector Lagrange Function as colour map of |u| + optional quiver.

    Parameters
    ----------
    u : dolfinx.fem.Function   value shape (2,); any Lagrange degree
    title : str
    quiver : bool              overlay subsampled arrows (default True)
    quiver_stride : int        take every N-th mesh node for arrows
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    msh = u.function_space.mesh
    gdim = msh.geometry.dim
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    npts = x.shape[0]

    # Interpolate to P1 vector so values align with geometry nodes
    V1v = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=(gdim,)))
    u1v = Function(V1v)
    u1v.interpolate(u)

    # Scatter: for a block-size-2 P1 space, dofmap.list gives block indices
    ldm = V1v.dofmap.list      # (ncells, 3)  — block indices
    arr = u1v.x.array.real     # length npts * gdim, interleaved per block
    ux = np.zeros(npts)
    uy = np.zeros(npts)
    ux[gdm.ravel()] = arr[ldm.ravel() * gdim + 0]
    uy[gdm.ravel()] = arr[ldm.ravel() * gdim + 1]
    mag = np.sqrt(ux**2 + uy**2)

    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, mag, shading="gouraud", cmap="viridis")
    plt.colorbar(tc, ax=ax, label="|u|")
    if quiver:
        idx = np.arange(0, npts, quiver_stride)
        ax.quiver(x[idx, 0], x[idx, 1], ux[idx], uy[idx],
                  color="white", alpha=0.6, width=0.002, scale_units="xy", scale=2.0)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()
# --- end canonical: plot_helpers ---

In [ ]:
# --- canonical: xdmf_series v1 ---
import sys
import pathlib
import basix.ufl as _bufl
from mpi4py import MPI
from dolfinx.io import XDMFFile
from dolfinx.fem import functionspace, Function


def _to_p1(f):
    """Interpolate f to P1 (scalar or vector) if needed."""
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    vshape = el.reference_value_shape
    if el.degree == 1:
        return f
    if vshape == ():
        V1 = functionspace(msh, ("Lagrange", 1))
    else:
        V1 = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=vshape))
    f1 = Function(V1, name=f.name)
    f1.interpolate(f)
    return f1


class XDMFSeries:
    """Time-series XDMF writer.

    Opens one XDMFFile, writes the mesh once on the first write() call,
    then appends each time step.  Higher-degree functions are automatically
    interpolated to P1 before writing.

    Usage::

        xdmf = XDMFSeries("output/solution.xdmf")
        for t in ...:
            xdmf.write([u1, p1, w1], t)
        xdmf.close()          # or: `with XDMFSeries(...) as xdmf:`

    Parameters
    ----------
    path : str or Path
        Output ``.xdmf`` file path.  The ``.h5`` sidecar is written alongside.
    download : bool, optional
        If True *and* running on Colab, tar the pair and trigger a download
        when :meth:`close` is called.
    """

    def __init__(self, path, download=False):
        self._path = pathlib.Path(path)
        self._path.parent.mkdir(parents=True, exist_ok=True)
        self._download = download
        self._xdmf = XDMFFile(MPI.COMM_WORLD, str(self._path), "w")
        self._mesh_written = False

    def write(self, funcs, t):
        """Write *funcs* at time *t*."""
        funcs_p1 = [_to_p1(f) for f in funcs]
        if not self._mesh_written:
            self._xdmf.write_mesh(funcs_p1[0].function_space.mesh)
            self._mesh_written = True
        for f in funcs_p1:
            self._xdmf.write_function(f, t)

    def close(self):
        """Close the file and optionally trigger a Colab download."""
        self._xdmf.close()
        if self._download and "google.colab" in sys.modules:
            import tarfile
            from google.colab import files as colab_files
            tar_path = str(self._path.with_suffix(".tar.gz"))
            with tarfile.open(tar_path, "w:gz") as tar:
                tar.add(str(self._path), arcname=self._path.name)
                h5 = self._path.with_suffix(".h5")
                if h5.exists():
                    tar.add(str(h5), arcname=h5.name)
            colab_files.download(tar_path)

    def __enter__(self):
        return self

    def __exit__(self, *_):
        self.close()
# --- end canonical: xdmf_series ---

In [ ]:
import numpy as np
import time
from mpi4py import MPI
from dolfinx import fem
from dolfinx.mesh import create_rectangle, CellType, DiagonalType
from dolfinx.fem import functionspace, Function, Constant, form, assemble_scalar
from dolfinx.fem.petsc import (
    assemble_matrix, assemble_vector,
    create_matrix, create_vector,
)
from petsc4py import PETSc
import ufl
import basix.ufl as bufl
import scipy.linalg.lapack as la
import matplotlib.pyplot as plt

# **Introduction**

Periodic boundary conditions model flows where the solution is spatially periodic in one direction. For a channel with periodicity in $x$, the solution at $x=0$ equals the solution at $x=L$.

Since DOLFINx does not natively support periodic BCs, we implement them algebraically via a *restriction matrix* $P\in\mathbb{R}^{n_{\text{full}}\times n_{\text{red}}}$. Columns of $P$ correspond to independent DOFs: right-boundary (source) DOFs are identified with their left-boundary (target) counterparts. Solving the reduced system $P^T A P\,x_r = P^T b$ and expanding via $x = P x_r$ automatically enforces periodicity.

# **Method**

**Define domain and mesh**

In [ ]:
# Domain: L x H rectangle, periodic in x
L = 2.0
H = 1.0

# Structured mesh: L*resolution x H*resolution cells, legacy default diagonal
resolution = 32
msh = create_rectangle(MPI.COMM_WORLD, [[0.0, 0.0], [L, H]],
                       [int(L*resolution), int(H*resolution)],
                       CellType.triangle, diagonal=DiagonalType.right)
tdim = msh.topology.dim
fdim = tdim - 1
n_cells = msh.topology.index_map(tdim).size_local

plot_mesh(msh)

**Define finite element approximation space**

In [ ]:
# P1 vector velocity and P1 scalar pressure
P1v = bufl.element("Lagrange", msh.basix_cell(), 1, shape=(2,))
P1s = bufl.element("Lagrange", msh.basix_cell(), 1)
V = functionspace(msh, P1v)
Q = functionspace(msh, P1s)

u = ufl.TrialFunction(V); p = ufl.TrialFunction(Q)
v = ufl.TestFunction(V);  q = ufl.TestFunction(Q)

**Define periodic restriction and boundary conditions**

In [ ]:
# ── Periodic restriction in x ──────────────────────────────────────────────
# P_V (n_full_V x n_red_V): identifies right-boundary dofs with left-boundary dofs.
# Algebraically: the reduced system has no right-boundary unknowns.
P_V, n_V = periodic_restriction(V, x_target=0.0, x_source=L)
P_Q, n_Q = periodic_restriction(Q, x_target=0.0, x_source=L)

# Pre-compute full->reduced dof maps (O(n_full) each, done once)
f2r_V = dof_map(P_V)
f2r_Q = dof_map(P_Q)

print(f"V: {V.dofmap.index_map.size_local*V.dofmap.bs} full dofs -> {n_V} reduced")
print(f"Q: {Q.dofmap.index_map.size_local}              full dofs -> {n_Q} reduced")

# ── Dirichlet BCs: u=(-1,0) at y=0, u=(1,0) at y=H ────────────────────────
V0, _m0 = V.sub(0).collapse(); m0 = np.array(_m0).ravel()
V1, _m1 = V.sub(1).collapse(); m1 = np.array(_m1).ravel()
coords_V = V.tabulate_dof_coordinates()    # (n_nodes, 3)
tol_bc   = 1e-12

lower_nodes = np.where(np.abs(coords_V[:, 1])       < tol_bc)[0]
upper_nodes  = np.where(np.abs(coords_V[:, 1] - H)  < tol_bc)[0]

# Full scalar dofs and prescribed values
_bc_full_dofs = np.concatenate([m0[lower_nodes], m1[lower_nodes],
                                 m0[upper_nodes], m1[upper_nodes]])
_bc_full_vals = np.concatenate([-np.ones(len(lower_nodes)),   # u_x = -1 at y=0
                                  np.zeros(len(lower_nodes)),  # u_y =  0 at y=0
                                  np.ones(len(upper_nodes)),   # u_x = +1 at y=H
                                  np.zeros(len(upper_nodes))]) # u_y =  0 at y=H

# Map to reduced dofs (deduplicate: corners appear in both periodic and Dirichlet)
_bc_rmap = {}
for _d, _v in zip(_bc_full_dofs, _bc_full_vals):
    _bc_rmap[int(f2r_V[_d])] = float(_v)
bc_rdofs_V = list(_bc_rmap.keys())
bc_rvals_V = list(_bc_rmap.values())

# **Results**

**Define method parameters**

In [ ]:
# Method parameters
nu    = 1.0e-3
T     = 80.0
plot_freq = 40

num_nnlin_iter = 5

# Time step: 0.25 * h_min (matching legacy dt = 0.25 * mesh.hmin())
dt   = time_step(msh, 1.0, C_CFL=0.25)
dt_c = Constant(msh, PETSc.ScalarType(dt))
nu_c = Constant(msh, PETSc.ScalarType(nu))

hmin = float(msh.h(tdim, np.arange(n_cells, dtype=np.int32)).min())
d3_c = Constant(msh, PETSc.ScalarType(1e-4 * hmin))   # pressure mass regularization

u0 = Function(V, name="u0"); u1 = Function(V, name="u1")
p0 = Function(Q, name="p0"); p1 = Function(Q, name="p1")

**Define variational problem**

In [ ]:
# GLS-stabilized Navier-Stokes, fractional step
# d1, d2: residual-based (override constant legacy values)
h_c   = ufl.CellDiameter(msh)
u_mag = ufl.sqrt(ufl.dot(u1, u1))
d1_c  = 1.0 / ufl.sqrt((1.0/dt_c)**2 + (u_mag/h_c)**2)
d2_c  = h_c * u_mag

um  = 0.5*(u + u0)
um1 = 0.5*(u1 + u0)

# Momentum form (residual)
Fu = (ufl.inner((u - u0)/dt_c + ufl.dot(ufl.grad(um), um1), v)*ufl.dx
      - p1*ufl.div(v)*ufl.dx
      + nu_c*ufl.inner(ufl.grad(um), ufl.grad(v))*ufl.dx
      + d1_c*ufl.inner((u - u0)/dt_c + ufl.dot(ufl.grad(um), um1) + ufl.grad(p1),
                       ufl.dot(ufl.grad(v), um1))*ufl.dx
      + d2_c*ufl.div(um)*ufl.div(v)*ufl.dx)
a_u = form(ufl.lhs(Fu)); l_u = form(ufl.rhs(Fu))

# Continuity form with pressure mass regularization d3*p*q (no pressure BC)
Fp = (d1_c*ufl.inner((u1 - u0)/dt_c + ufl.dot(ufl.grad(um1), um1) + ufl.grad(p),
                     ufl.grad(q))*ufl.dx
      + ufl.div(um1)*q*ufl.dx
      + d3_c*p*q*ufl.dx)
a_p = form(ufl.lhs(Fp)); l_p = form(ufl.rhs(Fp))
print("Forms compiled")

**Triple decomposition setup**

In [ ]:
import scipy.linalg.lapack as la

def triple_decomposition(grad_u):
    # dtype=float: legacy integer literal zeros truncate values (bug fix)
    new_grad = np.zeros((3, 3))
    for i in range(2):
        for j in range(2):
            new_grad[i, j] = grad_u[i, j]
    def dselect(arg1, arg2): return (arg2 == 0)
    T = la.dgees(dselect, new_grad, sort_t=1)[0]
    sh = np.linalg.norm([T[0, 1], T[0, 2], T[1, 2] + T[2, 1]])
    el = np.linalg.norm(np.diag(T))
    rr = np.sqrt(2 * min(abs(T[1, 2]), abs(T[2, 1]))**2)
    return sh, el, rr

# L2 projection setup for grad(u1) components (mass matrix assembled once)
_V2   = functionspace(msh, ("Lagrange", 1))
_u_s  = ufl.TrialFunction(_V2); _v_s = ufl.TestFunction(_V2)
_a_pr = form(_u_s * _v_s * ufl.dx)
_A_pr = create_matrix(_a_pr); assemble_matrix(_A_pr, _a_pr); _A_pr.assemble()
_kp   = PETSc.KSP().create(msh.comm)
_kp.setOperators(_A_pr); _kp.setType("cg"); _kp.getPC().setType("icc")
_kp.setTolerances(rtol=1e-10); _kp.setFromOptions()

_G00 = Function(_V2); _G01 = Function(_V2)
_G10 = Function(_V2); _G11 = Function(_V2)
_shear      = Function(_V2, name="shear")
_elongation = Function(_V2, name="elongation")
_rotation   = Function(_V2, name="rotation")
_divu       = Function(_V2, name="divu")

def _proj(expr, G):
    b = create_vector(_V2)
    with b.localForm() as loc: loc.set(0.0)
    assemble_vector(b, form(expr * _v_s * ufl.dx))
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    solve_checked(_kp, b, G.x.petsc_vec, "_kp"); G.x.scatter_forward()

def _do_triple_decomp():
    nG = ufl.nabla_grad(u1)
    _proj(nG[0,0],_G00); _proj(nG[0,1],_G01)
    _proj(nG[1,0],_G10); _proj(nG[1,1],_G11)
    g00=_G00.x.array.real; g01=_G01.x.array.real
    g10=_G10.x.array.real; g11=_G11.x.array.real
    n_v=_V2.dofmap.index_map.size_local
    for vx in range(n_v):
        gm=np.array([[g00[vx],g01[vx]],[g10[vx],g11[vx]]])
        sh,el,rr=triple_decomposition(gm)
        _shear.x.array[vx]=sh; _elongation.x.array[vx]=el; _rotation.x.array[vx]=rr
    _divu.x.array[:n_v]=g00[:n_v]+g11[:n_v]

**Set up I/O**

In [ ]:
import pathlib
pathlib.Path("results-PeriodicBC").mkdir(exist_ok=True)
xdmf = XDMFSeries("results-PeriodicBC/solution", download=False)
plot_time = 0.0

**Set up solvers**

In [ ]:
# --- canonical: ksp_helpers v1 ---
from petsc4py import PETSc


def make_ksp(A, ksp_type="bcgs", pc_type="ilu", amg=False, rtol=1e-6, atol=1e-14, max_it=500):
    ksp = PETSc.KSP().create(A.comm)
    ksp.setOperators(A)
    ksp.setType(ksp_type)
    pc = ksp.getPC()
    if amg:
        pc.setType("hypre")
        pc.setHYPREType("boomeramg")
    else:
        pc.setType(pc_type)
    ksp.setTolerances(rtol=rtol, atol=atol, max_it=max_it)
    ksp.setFromOptions()
    return ksp


def solve_checked(ksp, b, x, name):
    ksp.solve(b, x)
    reason = ksp.getConvergedReason()
    iters = ksp.getIterationNumber()
    if reason < 0:
        raise RuntimeError(
            f"KSP '{name}' diverged: reason={reason}, iterations={iters}"
        )
# --- end canonical: ksp_helpers ---

In [ ]:
# Full-space matrices (pre-allocated, reused with zeroEntries each iteration)
A_u = create_matrix(a_u); b_u = create_vector(V)
A_p = create_matrix(a_p); b_p = create_vector(Q)

# Reduced solution vectors
x_r_V = PETSc.Vec().createSeq(n_V, comm=PETSc.COMM_SELF)
x_r_Q = PETSc.Vec().createSeq(n_Q, comm=PETSc.COMM_SELF)

# Build initial reduced matrices to set up KSPs (u1=0 initial condition)
A_u.zeroEntries(); assemble_matrix(A_u, a_u, bcs=[]); A_u.assemble()
A_p.zeroEntries(); assemble_matrix(A_p, a_p, bcs=[]); A_p.assemble()

ksp_u = make_ksp(A_u.ptap(P_V))
ksp_p = make_ksp(A_p.ptap(P_Q), amg=True)

**Time stepping**

In [ ]:
t_wall0 = time.time()
t = dt; step = 0

while t < T + 1e-10:

    for _ in range(num_nnlin_iter):
        # ── Velocity ──
        A_u.zeroEntries(); assemble_matrix(A_u, a_u, bcs=[]); A_u.assemble()
        with b_u.localForm() as loc: loc.set(0.0)
        assemble_vector(b_u, l_u)
        b_u.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        Ar_V, br_V = reduce_system(A_u, b_u, P_V)
        apply_dirichlet_reduced(Ar_V, br_V, bc_rdofs_V, bc_rvals_V)
        ksp_u.setOperators(Ar_V, Ar_V); solve_checked(ksp_u, br_V, x_r_V, "ksp_u")
        u1.x.array[:] = expand_solution(x_r_V, P_V).getArray()
        u1.x.scatter_forward()

        # ── Pressure ──
        A_p.zeroEntries(); assemble_matrix(A_p, a_p, bcs=[]); A_p.assemble()
        with b_p.localForm() as loc: loc.set(0.0)
        assemble_vector(b_p, l_p)
        b_p.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        Ar_Q, br_Q = reduce_system(A_p, b_p, P_Q)
        ksp_p.setOperators(Ar_Q, Ar_Q); solve_checked(ksp_p, br_Q, x_r_Q, "ksp_p")
        p1.x.array[:] = expand_solution(x_r_Q, P_Q).getArray()
        p1.x.scatter_forward()

    if t > plot_time:
        s = "Time t = " + repr(t)
        print(s)

        xdmf.write([u1, p1], t)

        _do_triple_decomp()

        # Triple decomposition plot
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for _ax, _fn, _ttl in zip(axes,
                [_shear, _elongation, _rotation],
                ["Shear", "Strain", "Rotation"]):
            plot_scalar(_fn, title=_ttl, ax=_ax)
        plt.tight_layout(); plt.show()

        # Divergence
        fig2, ax2 = plt.subplots(figsize=(8, 3))
        plot_scalar(_divu, title="divergence", ax=ax2)
        plt.tight_layout(); plt.show()

        # Velocity and pressure
        plot_vector(u1, title="Velocity")
        plot_scalar(p1, title="Pressure")
        plt.show()

        plot_time += T / plot_freq

    u0.x.array[:] = u1.x.array
    t += dt; step += 1

xdmf.close()
t_wall = time.time() - t_wall0
print(f"Done: {step} steps, wall time = {t_wall:.1f}s")

**QoIs**

In [ ]:
norm_u1 = float(msh.comm.allreduce(
    assemble_scalar(form(ufl.inner(u1, u1)*ufl.dx)), op=MPI.SUM))**0.5
norm_p1 = float(msh.comm.allreduce(
    assemble_scalar(form(p1*p1*ufl.dx)), op=MPI.SUM))**0.5
print(f"||u1|| = {norm_u1:.6f}")
print(f"||p1|| = {norm_p1:.6f}")
print(f"cells  = {n_cells}")
print(f"dt     = {dt:.6f}")
print(f"steps  = {step}")
print(f"QOI norm_u = {norm_u1:.6g}")
print(f"QOI norm_p = {norm_p1:.6g}")

# **Discussion**

The periodic boundary condition is enforced by the restriction matrix $P$, which algebraically identifies right-boundary DOFs with their left-boundary counterparts. The reduced system $P^T AP$ is assembled by `ptap` (PETSc MatPtAP) and solved with BiCGSTAB + ILU (velocity) and BiCGSTAB + BoomerAMG (pressure). The pressure mass term $\delta_3 pq$ regularizes the pressure system in the absence of a Dirichlet pressure BC. Dirichlet conditions at the upper and lower walls ($u=(-1,0)$ at $y=0$, $u=(1,0)$ at $y=H$) are applied directly in the reduced system via `zeroRowsColumns`. Corner nodes that lie on both the periodic boundary and a Dirichlet wall are handled automatically: both the source and target corner DOFs map to the same reduced DOF, which receives the (consistent) Dirichlet value.